# Testing VertexAISearchSummaryTool Parameters

This notebook tests different configurations of the Vertex AI Search tool to understand:
- How citations work
- Different response formats
- Extractive answers
- What metadata we can extract


## Setup


In [1]:
import sys
from pathlib import Path
from pprint import pprint

# Add backend to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv

load_dotenv(project_root / ".env")

from langchain_google_community.vertex_ai_search import VertexAISearchSummaryTool

# Get config
PROJECT_ID = "nedo-ai-safety-citadel-ai"
DATA_STORE_ID = "periodic-pdf_1760448175058_gcs_store"

print(f"✅ Project ID: {PROJECT_ID}")
print(f"✅ Data Store ID: {DATA_STORE_ID[:50]}...")

✅ Project ID: nedo-ai-safety-citadel-ai
✅ Data Store ID: periodic-pdf_1760448175058_gcs_store...


## Test Query


In [7]:
TEST_QUERY = "Am I eligible for national health insurance? (Context: Designated Activities visa type)"
print(f"Test Query: {TEST_QUERY}")

Test Query: Am I eligible for national health insurance? (Context: Designated Activities visa type)


## 1. Default Configuration (Baseline)


In [24]:
tool_default = VertexAISearchSummaryTool(
    project_id=PROJECT_ID,
    data_store_id=DATA_STORE_ID,
    location_id="global",
    engine_data_type=0,
    summary_result_count=5,
    summary_include_citations=True,
    name="Default Config",
    description="Search for information about official procedures in Japan",
)

print("🔍 Testing DEFAULT configuration...")
result_default = tool_default.invoke(TEST_QUERY)
print(f"\nResult Type: {type(result_default)}")
print(f"Result Length: {len(str(result_default))} chars")
print("\n" + "=" * 80)
print("RESULT:")
print("=" * 80)
pprint(result_default)

/Users/tapatun/nedo-ai-safety-agent-new/venv/lib/python3.11/site-packages/langchain_google_community/vertex_ai_search.py:365: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


🔍 Testing DEFAULT configuration...


E0000 00:00:1760497587.684452 8256257 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.



Result Type: <class 'str'>
Result Length: 481 chars

RESULT:
('Generally, foreigners listed on a Certificate of Residence must enroll in '
 'National Health Insurance, with some exceptions [1]. One exception is people '
 'who have "Designated Activities" visa status, and who are receiving medical '
 'care, or taking care of those people who are receiving medical care [1]. '
 'Another exception is people who have “Designated Activities” visa status, '
 'residing in Japan for less than 1 year for the purpose of tourism, '
 'recreation, etc., and their spouse [1].\n')


## 2. With Extractive Answers


In [43]:
%autoreload 2

In [ ]:
class VertexAISearchSummaryToolExtended(VertexAISearchSummaryTool):
    def _run(self, user_query: str) -> str:
        """Runs the tool.
        Args:
            search_query: The query to run by the agent.
        Returns:
            The response from the agent. See https://cloud.google.com/generative-ai-app-builder/docs/reference/rpc/google.cloud.discoveryengine.v1beta#google.cloud.discoveryengine.v1beta.SearchRequest.ContentSearchSpec.SummarySpec
        """
        request = self._create_search_request(user_query)
        response = self._client.search(request)
        # https://cloud.google.com/generative-ai-app-builder/docs/reference/rpc/google.cloud.discoveryengine.v1beta#summary
        return (
            response.summary.summary_with_metadata.summary,
            response.summary.references,
        )

In [64]:
tool = VertexAISearchSummaryToolExtended(
    project_id=PROJECT_ID,
    data_store_id=DATA_STORE_ID,
    location_id="global",
    engine_data_type=0,  # Untructured data source
    # Quality (10 results, semantic chunks, high relevance)
    summary_result_count=5,
    # Citations (inline + extractive segments, not answers)
    summary_include_citations=True,
    get_extractive_answers=False,
    max_extractive_segment_count=3,
    return_extractive_segment_score=True,
    num_previous_segments=1,
    num_next_segments=1,
    # Safety (production-ready)
    summary_spec_kwargs={
        "ignore_adversarial_query": True,
        "ignore_jail_breaking_query": True,
        "ignore_low_relevant_content": True,
        "language_code": "en",
        "use_semantic_chunks": True,
    },
    # Format (if you need structured output)
    # response_format="content_and_artifact",
    name="Safety Filters",
    description="Search for information about official procedures in Japan",
)

print("🔍 Testing FASDF configuration...")
result_ = tool.invoke(TEST_QUERY)
print("\n" + "=" * 80)
print("RESULT:")
print("=" * 80)
pprint(result_)

/Users/tapatun/nedo-ai-safety-agent-new/venv/lib/python3.11/site-packages/langchain_google_community/vertex_ai_search.py:365: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


🔍 Testing FASDF configuration...
AAA
{'extractive_content_spec': max_extractive_segment_count: 3
return_extractive_segment_score: true
num_previous_segments: 1
num_next_segments: 1
}


E0000 00:00:1760501732.841029 8256257 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.



RESULT:
SearchPager<results {
  id: "451d8dd87a3aabb08829bbab8cbbf3b4"
  document {
    name: "projects/634361342501/locations/global/collections/default_collection/dataStores/periodic-pdf_1760448175058_gcs_store/branches/0/documents/451d8dd87a3aabb08829bbab8cbbf3b4"
    id: "451d8dd87a3aabb08829bbab8cbbf3b4"
    derived_struct_data {
      fields {
        key: "title"
        value {
          string_value: "Microsoft Word - 01_åł½æ°‚å†¥åº·ä¿šéŽºï¼‹ã†ﬁã†‘ã†»ï¼›ã†®ã†Šã†‘ã†¿ï¼‹è‰±èªžï¼›.docx"
        }
      }
      fields {
        key: "source_type"
        value {
          string_value: "gcs"
        }
      }
      fields {
        key: "link"
        value {
          string_value: "gs://citadel-ai-nedo-test-rag/Copy of kokuhonoshikumi_english.pdf"
        }
      }
      fields {
        key: "extractive_segments"
        value {
          list_value {
            values {
              struct_value {
                fields {
                  key: "relevanceScore"
            

In [ ]:
from langchain_google_community.vertex_ai_search import VertexAISearchSummaryTool

tool_extractive = VertexAISearchSummaryToolExtended(
    project_id=PROJECT_ID,
    data_store_id=DATA_STORE_ID,
    location_id="global",
    engine_data_type=0,
    summary_result_count=1,
    summary_include_citations=True,
    max_extractive_segment_count=5,
    get_extractive_answers=False,
    return_extractive_segment_score=True,
    num_previous_segments=1,
    query_expansion_condition=2,
    name="Default Config",
    description="Search for information about official procedures in Japan",
)

print("🔍 Testing DEFAULT configuration...")
result_extractive = tool_extractive.invoke(TEST_QUERY)
print(f"\nResult Type: {type(result_extractive)}")
print(f"Result Length: {len(str(result_extractive))} chars")
print("\n" + "=" * 80)
print("RESULT:")
print("=" * 80)
pprint(result_extractive)

/Users/tapatun/nedo-ai-safety-agent-new/venv/lib/python3.11/site-packages/langchain_google_community/vertex_ai_search.py:365: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


🔍 Testing DEFAULT configuration...
AAA
{'extractive_content_spec': max_extractive_segment_count: 5
return_extractive_segment_score: true
num_previous_segments: 1
num_next_segments: 1
}


E0000 00:00:1760498359.387814 8256257 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.



Result Type: <class 'google.cloud.discoveryengine_v1.services.search_service.pagers.SearchPager'>
Result Length: 23886 chars

RESULT:
SearchPager<results {
  id: "451d8dd87a3aabb08829bbab8cbbf3b4"
  document {
    name: "projects/634361342501/locations/global/collections/default_collection/dataStores/periodic-pdf_1760448175058_gcs_store/branches/0/documents/451d8dd87a3aabb08829bbab8cbbf3b4"
    id: "451d8dd87a3aabb08829bbab8cbbf3b4"
    derived_struct_data {
      fields {
        key: "title"
        value {
          string_value: "Microsoft Word - 01_åł½æ°‚å†¥åº·ä¿šéŽºï¼‹ã†ﬁã†‘ã†»ï¼›ã†®ã†Šã†‘ã†¿ï¼‹è‰±èªžï¼›.docx"
        }
      }
      fields {
        key: "source_type"
        value {
          string_value: "gcs"
        }
      }
      fields {
        key: "link"
        value {
          string_value: "gs://citadel-ai-nedo-test-rag/Copy of kokuhonoshikumi_english.pdf"
        }
      }
      fields {
        key: "extractive_segments"
        value {
          list_value {
 